In [4]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "Fri" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w6" / "Fri"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "Mon" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : d:\hanwha-agent


In [5]:
import shutil                       

from sqlalchemy import func, select

from app.db.seed_data import DOCUMENTS      
from app.db.session import session_scope
from app.models.document import Chunk, DocumentVersion
from app.services import document_service

MATERIALS = next((p for p in (SAMPLES, ROOT.parent / "seed_docs") if p.exists()), None)
if MATERIALS is None:                    
    print(" 재료 폴더가 없습니다 — sandbox\\w5\\day01\\samples 도, 프로젝트 폴더 옆 seed_docs 도 없습니다.")
    print("   이미 uploads/ 에 있는 파일만 적재합니다. \n")

rows = []
for doc in DOCUMENTS:
    for v in doc["versions"]:
        dst = ROOT / v["file_path"]         
        if not dst.exists() and MATERIALS is not None:
            src = next(MATERIALS.glob(f"{doc['id']}_*_{v['version']}.{v['file_format']}"), None)
            if src is None:
                print(f"  {doc['id']} {v['version']} : 재료가 없습니다 — 이 줄만 건너뜁니다")
                continue
            dst.parent.mkdir(exist_ok=True)
            shutil.copy2(src, dst)           
        with session_scope() as s:
            dv = s.scalar(select(DocumentVersion).where(
                DocumentVersion.doc_id == doc["id"], DocumentVersion.version == v["version"]))
            real = s.scalar(select(func.count()).select_from(Chunk)
                            .where(Chunk.version_id == dv.id)) if dv else 0
        if dv is not None and real == 0 and dst.exists():
            document_service.ingest_document(doc_id=doc["id"], version=v["version"], path=str(dst))
        rows.append((doc["id"], v["version"], v["file_format"], v["status"], v["chunk_count"]))

total = table_chunks = 0
with session_scope() as s:
    for doc_id, version, fmt, status, claimed in sorted(rows):
        dv = s.scalar(select(DocumentVersion).where(
            DocumentVersion.doc_id == doc_id, DocumentVersion.version == version))
        real = s.scalar(select(func.count()).select_from(Chunk).where(Chunk.version_id == dv.id))
        total += real
        table_chunks += s.scalar(select(func.count()).select_from(Chunk)
                                 .where(Chunk.version_id == dv.id, Chunk.kind == "표"))
        print(f"{doc_id} {version:<5}{fmt:<6}{status:<5} 청크 {real:>3}   (시드가 적어 둔 값 {claimed:>3})")

print()
print(f"여덟 줄 · 청크 합계 {total}  (시드의 주장 합계 {sum(r[4] for r in rows)})  ·  표 청크 {table_chunks}")

09:45:31 INFO     sentence_transformers.base.model: No device provided, using cpu
09:45:31 INFO     sentence_transformers.base.model: Loading SentenceTransformer model from models\bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

09:45:33 INFO     app.integrations.local_embed: 로컬 임베딩 모델 준비: models\bge-m3/1024 차원


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

09:48:18 INFO     app.rag.store: 청크 적재: DOC-SE-003v2.2 - 20청크 - 20벡터


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

09:48:32 INFO     app.rag.store: 청크 적재: DOC-SE-011v1.0 - 21청크 - 21벡터


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

09:48:48 INFO     app.rag.store: 청크 적재: DOC-PM-005v2.0 - 17청크 - 17벡터
DOC-HR-002 v3.1 pdf   현행    청크  24   (시드가 적어 둔 값 112)
DOC-HR-014 v1.1 docx  만료    청크  24   (시드가 적어 둔 값  79)
DOC-HR-014 v2.0 docx  현행    청크  53   (시드가 적어 둔 값  84)
DOC-HR-021 v1.2 docx  현행    청크  21   (시드가 적어 둔 값  41)
DOC-PM-005 v2.0 pdf   현행    청크  17   (시드가 적어 둔 값  87)
DOC-PU-007 v4.0 pdf   현행    청크  25   (시드가 적어 둔 값 156)
DOC-SE-003 v2.2 pdf   현행    청크  20   (시드가 적어 둔 값  98)
DOC-SE-011 v1.0 docx  현행    청크  21   (시드가 적어 둔 값  33)

여덟 줄 · 청크 합계 205  (시드의 주장 합계 690)  ·  표 청크 15


In [6]:
from app.db.session import session_scope
from app.rag import embedder, store

QUESTION='출장갈 때 숙박비 얼마 나옴?'

query_vec=embedder.embed_query(QUESTION)
with session_scope() as s:
    hits=store.vector_search(s,query_vec,limit=12)

print(f'질문:{QUESTION}->{len(hits)}건')
print('#\t\t문서\t\t\t판\t\t상태\t\t등급\t\t점수\t\t조각')
for i,h in enumerate(hits,1):
    print(f"{i}{h['title']}{h['version']}{h['version_status']} {h['security_level']}  {h['score']}  {h['locator']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

질문:출장갈 때 숙박비 얼마 나옴?->12건
#		문서			판		상태		등급		점수		조각
1국내출장 여비 규정v2.0현행 일반  0.6669  제3장 제2절 제14조(숙박비) · 별표1
2국내출장 여비 규정v1.1만료 일반  0.6474  제3장 제9조(식비)
3국내출장 여비 규정v2.0현행 일반  0.6398  제3장 제1절 제12조(식비) · 별표1
4국내출장 여비 규정v2.0현행 일반  0.6271  제4장 제22조(장기 출장)
5국내출장 여비 규정v2.0현행 일반  0.6266  제2장 제6조(출장의 구분)
6국내출장 여비 규정v2.0현행 일반  0.6249  표2
7국내출장 여비 규정v2.0현행 일반  0.6215  제3장 제2절 제16조(숙박비의 예외)
8국내출장 여비 규정v2.0현행 일반  0.6172  제3장 제1절 제11조(일비) · 별표1
9국내출장 여비 규정v2.0현행 일반  0.6147  표3
10국내출장 여비 규정v1.1만료 일반  0.6087  제3장 제11조(숙박비) · 별표1
11국내출장 여비 규정v1.1만료 일반  0.606  제2장 제7조(출장 승인)
12국내출장 여비 규정v1.1만료 일반  0.6057  제2장 제5조(출장의 구분)


In [8]:
from app.models.org import CLEARANCE

MJ={'name':'김민준', 'clearance':'일반'}
mine=CLEARANCE[MJ['clearance']]

print(f'# 문서\t등급 name {MJ['name']} {MJ['clearance']}')
for i,h in enumerate(hits,1):
    passed=CLEARANCE[h['security_level']] <= mine
    print(f'{i} {h['title']} {h['security_level']} {'통과' if passed else '차단'}')

blocked=[h for h in hits if CLEARANCE[h['security_level']]>mine]
print()
print(f'차단 건 수: {len(blocked)}건')

# 문서	등급 name 김민준 일반
1 국내출장 여비 규정 일반 통과
2 국내출장 여비 규정 일반 통과
3 국내출장 여비 규정 일반 통과
4 국내출장 여비 규정 일반 통과
5 국내출장 여비 규정 일반 통과
6 국내출장 여비 규정 일반 통과
7 국내출장 여비 규정 일반 통과
8 국내출장 여비 규정 일반 통과
9 국내출장 여비 규정 일반 통과
10 국내출장 여비 규정 일반 통과
11 국내출장 여비 규정 일반 통과
12 국내출장 여비 규정 일반 통과

차단 건 수: 0건


In [9]:
from app.repositories import document_repo

with session_scope() as s:
    document=document_repo.get_document(s,'DOC-FI-009')
    print('versions:',[v.version for v in document.versions])
    print('current:',document.current.version)

versions: ['1.5', 'v1.4', 'v1.6']
current: 1.5


In [10]:
from datetime import timedelta
from app.models import Document
with session_scope() as s:
    for document in s.scalars(select(Document).order_by(Document.id)).all():
        currents = [v for v in document.versions if v.status == '현행']
        if len(currents) <= 1:
            print(f'{document.id} : 이미 현행이 하나입니다 — 바꿀 것이 없습니다')
            continue
        keep = max(currents, key=lambda v: v.version)
        dropped = []
        for v in currents:
            if v is keep:
                continue
            v.status = '만료'
            v.expires_at = keep.effective_from - timedelta(days=1)
            v.index_status = '보관'
            dropped.append(v.version)
        print(f'{document.id} : 현행 = {keep.version} · 만료로 바꾼 판 {len(dropped)}개')

DOC-FI-009 : 현행 = v1.6 · 만료로 바꾼 판 2개
DOC-FI-010 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-HR-002 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-HR-014 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-HR-021 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-PM-005 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-PU-007 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-SE-003 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
DOC-SE-011 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다
HW-007 : 이미 현행이 하나입니다 — 바꿀 것이 없습니다


In [12]:
from app.rag import policy

QUESTION=[
    '일비가 예전에는 얼마였지?',
    '여비 규정 v1.1과 v2.0의 차이',
    '개정 전 교통비 정산 방식은?',
    '숙박비 상한 얼마?',
    '출장 신청 며칠 전에 해야?'
]
for q in QUESTION:
    print(f'{q}:{policy.wants_past_version(q)}')

일비가 예전에는 얼마였지?:True
여비 규정 v1.1과 v2.0의 차이:True
개정 전 교통비 정산 방식은?:False
숙박비 상한 얼마?:False
출장 신청 며칠 전에 해야?:False


In [14]:
import importlib

importlib.reload(policy)

DFT=policy.DEFAULT

print('만료 버전:',policy.version_allowed('만료',policy=DFT,question='숙박비 얼마?'))
print('만료 버전:',policy.version_allowed('만료',policy=DFT,question='예전 숙박비 얼마?'))
print('만료 버전:',policy.clearance_allowed('일반','대외비',policy=DFT))
print('만료 버전:',policy.clearance_allowed('대외비','대외비',policy=DFT))
print('만료 버전:',policy.version_allowed('만료',policy=policy.SearchPolicy(current_only=False),question='숙박비 얼마?'))

만료 버전: False
만료 버전: True
만료 버전: False
만료 버전: True
만료 버전: True


In [ ]:
importlib.reload(policy)
USER = {'id': '2019-0412', 'name': '김민준', 'clearance': '일반', 'dept_id': 'INFRA2'}
rows = [
    {'title': '국내출장 여비 규정', 'version': 'v2.0', 'version_status': '현행', 'security_level': '일반'}, 
    {'title': '국내출장 여비 규정', 'version': 'v1.1', 'version_status': '만료', 'security_level': '일반'}, 
    {'title': '구매·계약 규정', 'version': 'v4.0', 'version_status': '현행', 'security_level': '대외비'}, 
    {'title': '정보보안 지침', 'version': 'v2.2', 'version_status': '만료', 'security_level': '대외비'}
    ]
kept, reasons = policy.filter_candidates(rows, user=USER, question='출장 갈 때 숙박비 얼마 나옴?')
print(f'통과한 후보 ({len(kept)})')
for r in kept:
    print(' ', r['title'], r['version'])
print()
print(f'걸러진 이유 ({len(reasons)})')
for why in reasons:
    print(' ', why)

통과한 후보 (1)
  국내출장 여비 규정 v2.0

걸러진 이유 (3)
  국내출장 여비 규정v1.1-만료버전
  구매·계약 규정-보안등급(대외비)차단
  정보보안 지침v2.2-만료버전


In [16]:
import app.core.config as config_mod
from app.rag import embedder, store
importlib.reload(config_mod)
settings = config_mod.get_settings()
print(f'임계값 {settings.retrieval_threshold} · top_k {settings.retrieval_top_k}')

"""질문 하나를 검색해 정책으로 거르고, 최고 점수가 임계값 위인지 아래인지 찍는다."""
def ask(question: str) -> None:
    vec = embedder.embed_query(question)
    with session_scope() as s:
        found = store.vector_search(s, vec, limit=settings.retrieval_top_k)
    kept, _ = policy.filter_candidates(found, user=USER, question=question)
    best = max((r['score'] for r in kept), default=0.0)
    print()
    print('질문 :', question)
    if best < settings.retrieval_threshold:
        print(f'  최고 점수 {best:.2f} / 임계값 {settings.retrieval_threshold}  →  근거 부족')
    else:
        print(f'  최고 점수 {best:.2f} / 임계값 {settings.retrieval_threshold}  →  상위 세 줄')
        for r in kept[:3]:
            print(f'    {r['title']} {r['version']} {r['locator']}')
ask('출장 갈 때 숙박비 얼마 나오나요?')
ask('반려동물 동반 출장이 되나요?')

임계값 0.55 · top_k 12


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


질문 : 출장 갈 때 숙박비 얼마 나오나요?
  최고 점수 0.69 / 임계값 0.55  →  상위 세 줄
    국내출장 여비 규정 v2.0 제3장 제2절 제14조(숙박비) · 별표1
    국내출장 여비 규정 v2.0 제3장 제1절 제12조(식비) · 별표1
    국내출장 여비 규정 v2.0 표2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


질문 : 반려동물 동반 출장이 되나요?
  최고 점수 0.48 / 임계값 0.55  →  근거 부족
